# Can we predict next-month account churn?
Target: `churn_next_month`. Each row is an account-month snapshot; accounts repeat across rows, and the model is meant for future months.

**Design decisions (from checklist.md and columns.csv)**
- `account_ref` is an identifier: dropped (used only for grouping).
- `lifetime_months` comes from a CRM export of 2026-01-05 and counts total months the account was active, so it encodes the outcome (a cancelled account's lifetime ends at cancellation). It is not known at prediction time: **dropped**.
- `snapshot_month` is not used as a feature (the model must work for later months); it is used only for splitting.
- Test set = the latest months (2025-08 to 2025-11) *and* whole accounts: any account with a row in the test months is removed from the training rows entirely.
- Cross-validation inside training: expanding-window folds (earlier months train, later months validate), with accounts in the validation fold removed from that fold's training rows.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
df = pd.read_csv('data.csv')
print(df.shape, df.isna().sum().sum(), 'missing values')
print('churn rate', df.churn_next_month.mean().round(3))

## 1. Test set, set aside before any modelling

In [ ]:
TEST_START = '2025-08'
test = df[df.snapshot_month >= TEST_START].copy()
train = df[(df.snapshot_month < TEST_START) & ~df.account_ref.isin(test.account_ref)].copy()
assert set(train.account_ref).isdisjoint(test.account_ref) and train.snapshot_month.max() < test.snapshot_month.min()
print('train', train.shape, train.account_ref.nunique(), 'accounts | test', test.shape, test.account_ref.nunique(), 'accounts')
print('train churn', train.churn_next_month.mean().round(3))

## 2. Candidates, named before scoring anything
1. `baseline`: predicts the training churn rate for everyone (trivial).
2. `logreg`: standardised numeric features + one-hot plan, logistic regression, C in {0.01, 0.1, 1, 10}.
3. `hgb`: gradient boosting, (max_depth, learning_rate) in {(2,0.05),(3,0.05),(3,0.1)}, 150 iterations.

Features for 2-3: plan, seats, monthly_fee, tenure_months, usage_hours, support_tickets. Model selection metric: average precision (churn is the minority class); ROC-AUC reported too. The decision threshold is chosen on pooled out-of-fold predictions to maximise F1. Preprocessing lives inside the pipelines so it is fitted within each fold.

In [ ]:
FEATS = ['plan','seats','monthly_fee','tenure_months','usage_hours','support_tickets']
NUM = [f for f in FEATS if f != 'plan']
def prep(): return ColumnTransformer([('n', StandardScaler(), NUM), ('c', OneHotEncoder(), ['plan'])])
def prep_tree(): return ColumnTransformer([('n','passthrough',NUM), ('c', OneHotEncoder(), ['plan'])])
cands = {'baseline': make_pipeline(prep(), DummyClassifier(strategy='prior'))}
for C_ in [0.01,0.1,1,10]:
    cands[f'logreg C={C_}'] = make_pipeline(prep(), LogisticRegression(C=C_, max_iter=1000))
for d,lr in [(2,0.05),(3,0.05),(3,0.1)]:
    cands[f'hgb depth={d} lr={lr}'] = make_pipeline(prep_tree(), HistGradientBoostingClassifier(max_depth=d, learning_rate=lr, max_iter=150, random_state=0))
print(list(cands))

## 3. Cross-validation inside the training rows (expanding window, whole accounts)

In [ ]:
months = sorted(train.snapshot_month.unique())
# 4 validation blocks of consecutive months: the latest 16 training months
blocks = np.array_split(months[-16:], 4)
def folds():
    for b in blocks:
        val = train[train.snapshot_month.isin(b)]
        tr = train[(train.snapshot_month < b[0]) & ~train.account_ref.isin(val.account_ref)]
        yield tr, val
for tr, val in folds(): print(len(tr), len(val), val.snapshot_month.min(), val.snapshot_month.max(), round(val.churn_next_month.mean(),3))
oof = {}; rows = []
for name, m in cands.items():
    ps, ys, aps, aucs = [], [], [], []
    for tr, val in folds():
        f = clone(m).fit(tr[FEATS], tr.churn_next_month)
        p = f.predict_proba(val[FEATS])[:,1]
        ps.append(p); ys.append(val.churn_next_month.values)
        aps.append(average_precision_score(ys[-1], p)); aucs.append(roc_auc_score(ys[-1], p))
    oof[name] = (np.concatenate(ps), np.concatenate(ys))
    rows.append((name, np.mean(aps), np.std(aps), np.mean(aucs)))
cv = pd.DataFrame(rows, columns=['model','cv_AP','AP_sd','cv_AUC']).set_index('model').round(3)
print(cv)

In [ ]:
best = cv.cv_AP.drop('baseline').idxmax()
p, y = oof[best]
ths = np.linspace(0.02, 0.8, 79)
f1s = [f1_score(y, p>=t) for t in ths]
THRESH = float(ths[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(THRESH,2), '| CV F1:', round(max(f1s),3), '| flag-all F1:', round(f1_score(y, np.ones_like(y)),3))

## 4. Choice recorded before touching the test set
The chosen model and threshold are those printed above (highest mean CV average precision among the candidates; threshold maximising pooled out-of-fold F1). Nothing is changed after this point.

## 5. Final fit on all training rows; single test score (the headline)

In [ ]:
final = clone(cands[best]).fit(train[FEATS], train.churn_next_month)
pt = final.predict_proba(test[FEATS])[:,1]; yt = test.churn_next_month.values
base = clone(cands['baseline']).fit(train[FEATS], train.churn_next_month)
print(f'TEST ROC-AUC {roc_auc_score(yt,pt):.3f} | AP {average_precision_score(yt,pt):.3f} (no-skill AP = churn rate = {yt.mean():.3f})')
print(f'at threshold {THRESH:.2f}: precision {precision_score(yt,pt>=THRESH):.3f}, recall {recall_score(yt,pt>=THRESH):.3f}, F1 {f1_score(yt,pt>=THRESH):.3f}')
print('baseline test AUC', round(roc_auc_score(yt, base.predict_proba(test[FEATS])[:,1]),3))

## Conclusion
**Headline (single test score, 2025-08 to 2025-11, unseen accounts): ROC-AUC 0.703, average precision 0.217 vs 0.093 for a no-skill model.** So the features carry some ranking signal, but it is modest. The CV-selected logistic regression (C=0.01) beat the baseline and gradient boosting in CV (AP 0.42 vs 0.33; AUC about 0.61), and the test AUC is higher than CV AUC, so treat these numbers as noisy.

The CV-chosen F1 threshold (0.08) is so low that the model flags essentially every account (recall 0.997, precision 0.094, i.e. the base rate), so as a hard yes/no classifier it adds nothing over "flag all". Use it for ranking accounts (e.g. contacting the top-scoring decile), not for thresholded decisions, unless a different operating point is agreed on CV.

Caveats: the monthly churn rate swings strongly over time (about 3-5% in spring 2024, over 50% in Dec 2024 - Jan 2025, then ~10%), so CV folds are a shifting benchmark and thresholds do not transfer well across periods. Removing accounts that appear in the test months from training reduces training data. Test rows from the same account are correlated, so test uncertainty is larger than the row count suggests. `lifetime_months` was deliberately dropped: it would look highly predictive but is unavailable at prediction time.